# Pre-training and Downstream Tasks
The achievements of transformers can largely be attributed to **pre-training**. Pre-training of a model separates common components from many neural network-based systems, and then trains them on huge amounts of unlabeled data using self-supervision. These pre-trained models serve as foundation models that can be easily adapted to different tasks via fine-tuning or prompting. 

In deep learning, pre-training refers to the process of optimizing a neural network before it is further trained/tuned and applied to the tasks of interest. This approach is based on an assumption that a model pre-trained on one task can be adapted to perform another task. This reduces the reliance on task-specific labeled data, enabling the development of more general models that are not confined to particular problems. So, unlike standard learning problems, pre-training does not assume specific downstream tasks to which the model will be applied. Instead, the goal is to train a model that can generalize across various tasks. Furthermore, To adapt the model to these tasks, we need to adjust the parameters slightly using labeled data or prompt the model with task descriptions.

For simplicity, we define a single model as follows:
$$o = g(x_0, x_1,\dots,x_N; \theta) = g_\theta(x_0, x_1,\dots,x_N)$$
where $x_0$ might represent a start token (e.g. [s] or [CLS]) and $x_1, ..., x_N$ are the input tokens. $g_\theta$ denotes a neural network architecture (e.g. transformer architecture) with parameters $\theta$. 

In definining some transformer architecture $g_\theta$, it's not immediately clear what the training process looks like. In particular, while we know that we are working with feed-forward networks and using backpropagation to update gradients, () what's the loss function? What are the labels? What does the output $o$ really even represent? It turns out that these decisions correspond to the specific nature of the problem. We will talk more about these in the **encoder** and **decoder** notebooks. For now, let's focus on the bigger picture.

## Self-supervised pre-training

During the resurgence of neural networks and machine learning, many early attempts focused on using **unsupervised learning** for pre-training. In these methods, the parameters of a neural network are optimized using a criterion that is not directly related to specific tasks. Another approach was to train a neural network on labeled data -- **supervised learning**. This can be achieved, for example, by including a classification layer to form a classification system.

The third and most prominent approach is a paradigm called **self-supervised** learning. In this approach, a neural network is trained using the supervision signals generated by itself, rather than those provided by
humans. In other words, the data serves as both the training example the label. Self-supervised pre-training
in NLP does not rely on an initial model for annotating the data. Instead, all the supervision sig-
nals are created from the text, and the entire model is trained from scratch. A well-known example
of this is training sequence models by successively predicting a masked word given its preceding
or surrounding words in a text.

## Fine-tuning and Prompting

The discussion of pre-training issues in NLP typically involves two types of problems: sequence modeling (or sequence encoding) and sequence generation (decoding). 
- **Sequence Encoding Models**: Given a sequence of words or tokens, a sequence encoding model represents this sequence as either a real-valued vector or a sequence of vectors, and obtains a representation of the sequence. This representation is typically used as input to another model, such as a sentence classification system.
- **Sequence Generation Models**: In NLP, sequence generation generally refers to the problem of generating a sequence of tokens based on a given context. The term context has different meanings across applications. For example, it refers to the preceding tokens in language modeling, and refers to the source-language sequence in machine translation.

We need different techniques for applying these models to downstream tasks after pre-training. We consider two possibilities: (1) **fine-tuning** and (2) **prompting**.

### Fine-tuning
For fine-tuning, we will focus on sequence encoding models. Consider a pre-trained transformer model with optimized parameters $\hat{\theta}$:
$$ H = g_{\hat{\theta}}(X) = \text{Encode}_{\hat{\theta}}(X) $$
Here, $H$ are real-valued vector outputs $\set{h_0, h_1, \dots, h_N}$. 

Consider a sentiment classification problem, where we want to determine if a piece of text has "positive," "neutral," or "negative" sentiment. We then stack a classification head on top of the encoder. To do this, let $\text{Classify}_\omega(\dot)$ be a neural network and let the text classification model be:
$$\mathbb{P}_{\hat{\theta}, \omega}(\cdot|X) = \text{Classify}_\omega(H) = \text{Classify}_\omega(\text{Encode}_{\hat{\theta}}(X))$$
Because the model parameters $\omega$ and $\hat{\theta}$ are not optimized for the classification task, we cannot directly use this model. Instead, we must use a modified version of the model that is adapted to
the task. A typical way is to **fine-tune** the model by giving explicit labeling in downstream tasks. In other words, we can train $\mathbb{P}_{\hat{\theta}, \omega}(\cdot|X)$ on a labeled dataset, treating it as a common supervised learning task. The outcome of the fine-tuning is the parameters $\tilde{\theta}$ and $\tilde{\omega}$. Alternatively, we can also fix the encoder parameters $\hat{\theta}$ to maintain their pre-trained state, and focus solely on optimizing $\omega$. This allows the classifier to be efficiently adapted to work in tandem with the pre-trained encoder.

In general, the amount of labeled data used in fine-tuning is small compared to that of the
pre-training data, and so fine-tuning is less computationally expensive. This makes the adaption
of pre-trained models very efficient in practice: given a pre-trained model and a downstream task,
we just need to collect some labeled data, and slightly adjust the model parameters on this data.

### Prompting
Unlike sequence encoding models, sequence generation models are often employed independently to address language generation problems, such as question answering and machine translation, without the need for additional modules. Thus, fine-tuning a model is pretty straightforward, as it only involves feeding the model the right context. By using instruction-based prompts, we can adapt large language models to solve NLP problems without the need for additional training.

This is also an example of the **zero-shot learning** capability of large language models, which can perform tasks that were not observed during the training phase. Another method for enabling new capabilities in a neural network is **few-shot learning**. This is typically achieved through in-context learning (ICT).

## Hyperparameters to consider

### Pre-training
1. Tokenization & Vocabulary
    - vocab_size
    -Tokenization scheme (BPE, WordPiece, Unigram)
    - Special tokens ([CLS], [MASK], <bos>, <eos>)
2. Context & Positional Modeling
    - max_sequence_length (context window)
    - Positional encoding type (absolute, RoPE, ALiBi)
    - Positional scaling / rotary base (for RoPE)
3. Pretraining Objective
    - Objective type:
        - MLM (BERT)
        - Autoregressive LM (GPT)
        - Prefix LM (T5)
    - Masking rate & strategy (MLM only)
    - Sentence-level objective: NSP, SOP, or none
4. Output Parameterization
    - Weight tying (embedding ↔ softmax)
    - Logit scaling before softmax

### Fine-tuning
1. Input Adaptation
    - Fine-tuning max_sequence_length (often ≤ pretraining max)
    - Truncation / sliding window strategy
2. Objective-Specific (Task Dependent)
    - Token-level loss vs sequence-level loss
    - Label smoothing (classification tasks)
    - Class weighting (imbalanced NLP tasks)
3. Generation-Time (Decoder Models Only - only affects inference, not training)
    - Temperature
    - Top-k
    - Top-p (nucleus sampling)
    - Repetition penalty
    - Beam width